# Topology depth–width trade-off: fixed rerun (Nov 2026)

**Setup (once):** Settings → Accelerator **None** (CPU; the GPU is not used by this code) and Internet **On**.

**Two stages.** Set `STAGE` in the next cell: run `"noisy"` first, then `"noiseless"`. Use *Save Version → Save & Run All (Commit)* so it runs in the background.

**If a stage stops at the time budget:** open the notebook, *Add Input → Your Work →* this notebook's last version, then *Save & Run All* again. Finished cells are copied in and skipped.

**When both stages are done:** download `outputs_fixed.zip` from the Output tab and send it back.

In [ ]:
STAGE = "noisy"        # "noisy" first, then "noiseless"
MAX_HOURS = 11.0       # stop cleanly before Kaggle's 12 h limit; rerun to continue
SMOKE = True         # True = tiny test grid (a few minutes); keep False for the real run

In [ ]:
!pip install -q 'pennylane==0.44.1'
import pennylane as qml, os
print('PennyLane', qml.__version__, '|', os.cpu_count(), 'CPUs')

In [ ]:
%%writefile ansatze.py
"""
ansatze.py
==========
Hardware-efficient ansatz topologies. Identical rotation layers
(single RY per qubit per layer) so depth and width are the only varying
factors. Topology only differs in the entangling block.

Topologies
----------
linear         : nearest-neighbour CNOT chain, no wrap-around.
ring           : nearest-neighbour CNOT chain + wrap-around.
brick          : alternating even-odd / odd-even CZ pattern.
all_to_all     : CNOT between every pair (i, j), i < j.
star           : qubit 0 is the hub, CNOT(0, j) for all j > 0.
heavy_hex      : LEGACY approximation (chain + (i, i+2) chords). Kept only so
                 old results can be reproduced. Not a real heavy-hex graph.
heavy_hex_true : connected n-qubit subgraph of the real IBM Falcon 27-qubit
                 heavy-hex coupling map (BFS order from qubit 0).

Every ansatz accepts an optional ``layer_hook``. It is called once at the end
of every layer (after the entangling block). metrics.py passes the noise
channel through it, so noise is applied per layer, as the paper states.

Parameter count: n_qubits * n_layers for all topologies.
"""

from collections import deque

import pennylane as qml


# ---------------------------------------------------------------------------
# Real IBM Falcon r4 27-qubit heavy-hex coupling map (undirected edges).
# Source: IBM 27-qubit Falcon backends (e.g. ibmq_montreal / ibmq_kolkata).
# ---------------------------------------------------------------------------
FALCON27_EDGES = [
    (0, 1), (1, 2), (1, 4), (2, 3), (3, 5), (4, 7), (5, 8), (6, 7),
    (7, 10), (8, 9), (8, 11), (10, 12), (11, 14), (12, 13), (12, 15),
    (13, 14), (14, 16), (15, 18), (16, 19), (17, 18), (18, 21), (19, 20),
    (19, 22), (21, 23), (22, 25), (23, 24), (24, 25), (25, 26),
]


def _falcon_subgraph(n_qubits, root=0):
    """Edges of the connected n-node subgraph of Falcon-27 reached by BFS from
    `root`, relabelled to wires 0..n-1. A BFS prefix of a connected graph is
    always connected."""
    adj = {}
    for a, b in FALCON27_EDGES:
        adj.setdefault(a, []).append(b)
        adj.setdefault(b, []).append(a)
    order, seen, queue = [], {root}, deque([root])
    while queue and len(order) < n_qubits:
        v = queue.popleft()
        order.append(v)
        for w in sorted(adj[v]):
            if w not in seen:
                seen.add(w)
                queue.append(w)
    if len(order) < n_qubits:
        raise ValueError(f"Falcon-27 has only {len(order)} qubits reachable; asked for {n_qubits}")
    label = {v: i for i, v in enumerate(order)}
    keep = set(order)
    return [(label[a], label[b]) for a, b in FALCON27_EDGES if a in keep and b in keep]


def _heavy_hex_gates(n_qubits):
    """LEGACY approximation used in the original sweeps (not real heavy-hex)."""
    gates = []
    for i in range(n_qubits - 1):
        gates.append((i, i + 1))
    for i in range(0, n_qubits - 2, 3):
        if i + 2 < n_qubits:
            gates.append((i, i + 2))
    seen, unique = set(), []
    for g in gates:
        key = (min(g), max(g))
        if key not in seen:
            seen.add(key)
            unique.append(g)
    return unique


# ---------------------------------------------------------------------------
# Entangling blocks
# ---------------------------------------------------------------------------
def _entangle(name, n_qubits, layer):
    if name == "linear":
        for q in range(n_qubits - 1):
            qml.CNOT(wires=[q, q + 1])
    elif name == "ring":
        for q in range(n_qubits - 1):
            qml.CNOT(wires=[q, q + 1])
        if n_qubits > 1:
            qml.CNOT(wires=[n_qubits - 1, 0])
    elif name == "brick":
        start = 0 if layer % 2 == 0 else 1
        for q in range(start, n_qubits - 1, 2):
            qml.CZ(wires=[q, q + 1])
    elif name == "all_to_all":
        for i in range(n_qubits):
            for j in range(i + 1, n_qubits):
                qml.CNOT(wires=[i, j])
    elif name == "star":
        for j in range(1, n_qubits):
            qml.CNOT(wires=[0, j])
    elif name == "heavy_hex":
        for c, t in _heavy_hex_gates(n_qubits):
            qml.CNOT(wires=[c, t])
    elif name == "heavy_hex_true":
        for c, t in _falcon_subgraph(n_qubits):
            qml.CNOT(wires=[c, t])
    else:
        raise ValueError(f"Unknown topology: {name}")


TOPOLOGIES = ["linear", "ring", "brick", "all_to_all", "star", "heavy_hex", "heavy_hex_true"]


def _rotation_layer(params, n_qubits, idx):
    """Single RY rotation on every qubit. Returns updated idx."""
    for q in range(n_qubits):
        qml.RY(params[idx], wires=q)
        idx += 1
    return idx


def _build(name, params, n_qubits, n_layers, x=None, layer_hook=None):
    idx = 0
    for layer in range(n_layers):
        if x is not None:
            for q in range(n_qubits):
                qml.RY(x[q], wires=q)
        idx = _rotation_layer(params, n_qubits, idx)
        _entangle(name, n_qubits, layer)
        if layer_hook is not None:
            layer_hook()


def _make(name):
    def ansatz(params, n_qubits, n_layers, layer_hook=None):
        _build(name, params, n_qubits, n_layers, layer_hook=layer_hook)
    ansatz.__name__ = f"ansatz_{name}"
    return ansatz


def _make_data(name):
    def ansatz(x, params, n_qubits, n_layers, layer_hook=None):
        _build(name, params, n_qubits, n_layers, x=x, layer_hook=layer_hook)
    ansatz.__name__ = f"ansatz_{name}_with_data"
    return ansatz


ANSATZE = {name: _make(name) for name in TOPOLOGIES}
ANSATZE_WITH_DATA = {name: _make_data(name) for name in TOPOLOGIES}

# Backward-compatible names
ansatz_linear = ANSATZE["linear"]
ansatz_ring = ANSATZE["ring"]
ansatz_brick = ANSATZE["brick"]
ansatz_all_to_all = ANSATZE["all_to_all"]
ansatz_star = ANSATZE["star"]
ansatz_heavy_hex = ANSATZE["heavy_hex"]
ansatz_linear_with_data = ANSATZE_WITH_DATA["linear"]
ansatz_ring_with_data = ANSATZE_WITH_DATA["ring"]
ansatz_brick_with_data = ANSATZE_WITH_DATA["brick"]


def n_params(n_qubits, n_layers, ansatz_name=None):
    """Total trainable parameters. Identical for all topologies."""
    return n_qubits * n_layers


In [ ]:
%%writefile noise.py
"""
noise.py
========
Noise model and device construction.

Two noise families:
    - Depolarizing: uniform mixing toward the maximally mixed state.
    - Amplitude damping: directional decay toward |0>. Physically dominant
      on superconducting hardware (T1 process).

Applied per-qubit at the end of every layer. The rate corresponds to total
error per qubit per layer.

Presets:
    noiseless              : no noise (statevector simulator).
    depolarizing_realistic : p = 1e-3 per qubit per layer (IBM Heron class).
    depolarizing_strong    : p = 1e-2 per qubit per layer (older NISQ).
    amplitude_realistic    : gamma = 1e-3 per qubit per layer.
    amplitude_strong       : gamma = 1e-2 per qubit per layer.

The channel `kind` and rate `rate` are queried by metrics.py to apply the
correct channel inline in the qnode.
"""

import pennylane as qml


NOISE_PRESETS = {
    "noiseless": {
        "kind": "none",
        "rate": 0.0,
    },
    "depolarizing_realistic": {
        "kind": "depolarizing",
        "rate": 1e-3,
    },
    "depolarizing_strong": {
        "kind": "depolarizing",
        "rate": 1e-2,
    },
    "amplitude_realistic": {
        "kind": "amplitude_damping",
        "rate": 1e-3,
    },
    "amplitude_strong": {
        "kind": "amplitude_damping",
        "rate": 1e-2,
    },
}


def make_device(n_qubits, noise_preset="noiseless"):
    cfg = NOISE_PRESETS[noise_preset]
    if cfg["kind"] == "none":
        return qml.device("default.qubit", wires=n_qubits)
    return qml.device("default.mixed", wires=n_qubits)


def is_noiseless(noise_preset):
    return NOISE_PRESETS[noise_preset]["kind"] == "none"


def apply_layer_noise(noise_preset, n_qubits):
    """
    Apply per-qubit noise channel for the configured preset.
    Call this at the end of every ansatz layer.
    """
    cfg = NOISE_PRESETS[noise_preset]
    kind = cfg["kind"]
    rate = cfg["rate"]

    if kind == "none":
        return
    if kind == "depolarizing":
        for q in range(n_qubits):
            qml.DepolarizingChannel(rate, wires=q)
    elif kind == "amplitude_damping":
        for q in range(n_qubits):
            qml.AmplitudeDamping(rate, wires=q)
    else:
        raise ValueError(f"Unknown noise kind: {kind}")


In [ ]:
%%writefile metrics.py
"""
metrics.py
==========
Metrics. Noise is injected at the end of EVERY layer through the ansatz
``layer_hook`` (fixed Nov 2026: it was previously applied once after the whole
circuit), so any noise preset registered in noise.py works automatically.

Metrics:
    haar_kl_divergence       : KL from the Haar fidelity distribution. Default
                               reference is the REAL (orthogonal) ensemble, since
                               RY + CNOT/CZ circuits only produce real states.
    frame_potential_ratio    : second frame potential / real-Haar value (>= 1;
                               -> 1 for a real 2-design). Does not saturate.
    gradient_variance_all    : Var of dC/dtheta_k averaged over ALL parameters,
                               with cost "z0" (<Z_0>) or "local" (mean_i <Z_i>).
    effective_kernel_rank    : effective rank of empirical kernel matrix.
    gradient_variance        : variance of d<Z_0>/dtheta_0 over random params.
    function_output_variance : variance of circuit output over random inputs.
    qfim_metrics             : QFIM trace and Abbas effective dimension.
"""

import itertools
import numpy as np
import pennylane as qml
from scipy.stats import entropy

from ansatze import ANSATZE, ANSATZE_WITH_DATA, n_params
from noise import make_device, NOISE_PRESETS, apply_layer_noise, is_noiseless


def _hook(noise_preset, n_qubits):
    """Per-layer noise hook, or None for noiseless runs."""
    if is_noiseless(noise_preset):
        return None
    return lambda: apply_layer_noise(noise_preset, n_qubits)


# ---------------------------------------------------------------------------
# QNode builders
# ---------------------------------------------------------------------------
def _make_state_qnode(ansatz_name, n_qubits, n_layers, noise_preset):
    ansatz = ANSATZE[ansatz_name]
    dev = make_device(n_qubits, noise_preset)

    if is_noiseless(noise_preset):
        @qml.qnode(dev)
        def circuit(params):
            ansatz(params, n_qubits, n_layers)
            return qml.state()
        return circuit, "pure"

    hook = _hook(noise_preset, n_qubits)

    @qml.qnode(dev)
    def circuit(params):
        ansatz(params, n_qubits, n_layers, layer_hook=hook)
        return qml.density_matrix(wires=range(n_qubits))
    return circuit, "mixed"


def _make_expval_qnode(ansatz_name, n_qubits, n_layers, noise_preset, observable=None):
    ansatz = ANSATZE[ansatz_name]
    dev = make_device(n_qubits, noise_preset)
    if observable is None:
        observable = qml.PauliZ(0)

    hook = _hook(noise_preset, n_qubits)

    @qml.qnode(dev, interface="autograd")
    def circuit(params):
        ansatz(params, n_qubits, n_layers, layer_hook=hook)
        return qml.expval(observable)
    return circuit


def _make_data_qnode(ansatz_name, n_qubits, n_layers, noise_preset):
    if ansatz_name not in ANSATZE_WITH_DATA:
        raise NotImplementedError(
            f"Data-encoding variant not implemented for {ansatz_name}. "
            f"Add it to ansatze.py if you need function-space metrics for this topology."
        )
    ansatz = ANSATZE_WITH_DATA[ansatz_name]
    dev = make_device(n_qubits, noise_preset)

    hook = _hook(noise_preset, n_qubits)

    @qml.qnode(dev, interface="autograd")
    def circuit(x, params):
        ansatz(x, params, n_qubits, n_layers, layer_hook=hook)
        return qml.expval(qml.PauliZ(0))
    return circuit


def _make_data_state_qnode(ansatz_name, n_qubits, n_layers, noise_preset):
    if ansatz_name not in ANSATZE_WITH_DATA:
        raise NotImplementedError(
            f"Data-encoding variant not implemented for {ansatz_name}."
        )
    ansatz = ANSATZE_WITH_DATA[ansatz_name]
    dev = make_device(n_qubits, noise_preset)

    if is_noiseless(noise_preset):
        @qml.qnode(dev)
        def circuit(x, params):
            ansatz(x, params, n_qubits, n_layers)
            return qml.state()
        return circuit

    hook = _hook(noise_preset, n_qubits)

    @qml.qnode(dev)
    def circuit(x, params):
        ansatz(x, params, n_qubits, n_layers, layer_hook=hook)
        return qml.density_matrix(wires=range(n_qubits))
    return circuit


# ---------------------------------------------------------------------------
# Haar reference
# ---------------------------------------------------------------------------
def _haar_fidelities_pure(n_qubits, n_pairs, rng):
    dim = 2 ** n_qubits
    fids = np.empty(n_pairs)
    for k in range(n_pairs):
        a = rng.normal(size=dim) + 1j * rng.normal(size=dim)
        a /= np.linalg.norm(a)
        b = rng.normal(size=dim) + 1j * rng.normal(size=dim)
        b /= np.linalg.norm(b)
        fids[k] = np.abs(np.vdot(a, b)) ** 2
    return fids


def _haar_fidelities_real(n_qubits, n_pairs, rng):
    """Fidelities between independent random REAL unit vectors (orthogonal
    ensemble). Distribution is Beta(1/2, (d-1)/2), mean 1/d."""
    dim = 2 ** n_qubits
    a = rng.normal(size=(n_pairs, dim))
    b = rng.normal(size=(n_pairs, dim))
    a /= np.linalg.norm(a, axis=1, keepdims=True)
    b /= np.linalg.norm(b, axis=1, keepdims=True)
    return np.einsum("ij,ij->i", a, b) ** 2


def haar_frame_potential(n_qubits, real=True):
    """Second frame potential E[F^2] of the Haar ensemble."""
    d = 2 ** n_qubits
    return 3.0 / (d * (d + 2)) if real else 2.0 / (d * (d + 1))


def _fidelity(state_a, state_b, mode):
    if mode == "pure":
        return float(np.abs(np.vdot(state_a, state_b)) ** 2)
    return float(np.real(np.trace(state_a @ state_b)))


# ---------------------------------------------------------------------------
# Metric 1: Haar KL
# ---------------------------------------------------------------------------
def haar_kl_divergence(ansatz_name, n_qubits, n_layers, noise_preset="noiseless",
                       n_samples=60, n_bins=50, seed=0, reference="real"):
    rng = np.random.default_rng(seed)
    qnode, mode = _make_state_qnode(ansatz_name, n_qubits, n_layers, noise_preset)
    n_p = n_params(n_qubits, n_layers)

    states = []
    for _ in range(n_samples):
        params = rng.uniform(0, 2 * np.pi, size=n_p)
        states.append(qnode(params))

    pairs = list(itertools.combinations(range(n_samples), 2))
    vqc_fids = np.array([_fidelity(states[i], states[j], mode) for i, j in pairs])
    if reference == "real":
        haar_fids = _haar_fidelities_real(n_qubits, len(pairs), rng)
    else:
        haar_fids = _haar_fidelities_pure(n_qubits, len(pairs), rng)

    edges = np.linspace(0, 1, n_bins + 1)
    h_vqc, _ = np.histogram(vqc_fids, bins=edges, density=False)
    h_haar, _ = np.histogram(haar_fids, bins=edges, density=False)
    h_vqc = h_vqc.astype(float) + 1e-10
    h_haar = h_haar.astype(float) + 1e-10
    h_vqc /= h_vqc.sum()
    h_haar /= h_haar.sum()
    return float(entropy(h_vqc, h_haar))


def frame_potential_ratio(ansatz_name, n_qubits, n_layers, noise_preset="noiseless",
                          n_samples=60, seed=0, reference="real"):
    """Second frame potential of the circuit ensemble divided by the Haar value.
    Ratio >= 1 (up to sampling error); 1 means the ensemble matches a 2-design.
    Unlike the binned KL, this does not saturate at large n."""
    rng = np.random.default_rng(seed)
    qnode, mode = _make_state_qnode(ansatz_name, n_qubits, n_layers, noise_preset)
    n_p = n_params(n_qubits, n_layers)
    states = [qnode(rng.uniform(0, 2 * np.pi, size=n_p)) for _ in range(n_samples)]
    fids = np.array([_fidelity(states[i], states[j], mode)
                     for i, j in itertools.combinations(range(n_samples), 2)])
    return float(np.mean(fids ** 2) / haar_frame_potential(n_qubits, real=(reference == "real")))


# ---------------------------------------------------------------------------
# Metric 2: Effective kernel rank
# ---------------------------------------------------------------------------
def effective_kernel_rank(ansatz_name, n_qubits, n_layers, noise_preset="noiseless",
                          n_data=20, seed=0):
    rng = np.random.default_rng(seed)
    qnode = _make_data_state_qnode(ansatz_name, n_qubits, n_layers, noise_preset)
    n_p = n_params(n_qubits, n_layers)
    params = rng.uniform(0, 2 * np.pi, size=n_p)
    X = rng.uniform(0, np.pi, size=(n_data, n_qubits))

    states = [qnode(x, params) for x in X]
    mode = "pure" if is_noiseless(noise_preset) else "mixed"

    K = np.zeros((n_data, n_data))
    for i in range(n_data):
        for j in range(n_data):
            K[i, j] = _fidelity(states[i], states[j], mode)
    K = (K + K.T) / 2

    tr = np.trace(K)
    tr_sq = np.trace(K @ K)
    if tr_sq <= 0:
        return 0.0
    return float(tr ** 2 / tr_sq)


# ---------------------------------------------------------------------------
# Metric 3: Gradient variance
# ---------------------------------------------------------------------------
def gradient_variance(ansatz_name, n_qubits, n_layers, noise_preset="noiseless",
                      n_samples=50, seed=0, use_data=False):
    """
    Variance of d<Z_0>/dtheta_0 over random parameter samples.
    Standard barren-plateau diagnostic (McClean 2018, Cerezo 2021).
    """
    from pennylane import numpy as pnp

    rng = np.random.default_rng(seed)
    n_p = n_params(n_qubits, n_layers)

    if use_data:
        qnode = _make_data_qnode(ansatz_name, n_qubits, n_layers, noise_preset)
        x_fixed = rng.uniform(0, np.pi, size=n_qubits)
        def cost(p):
            return qnode(x_fixed, p)
    else:
        qnode = _make_expval_qnode(ansatz_name, n_qubits, n_layers, noise_preset)
        def cost(p):
            return qnode(p)

    grad_fn = qml.grad(cost, argnums=0)

    g0_samples = []
    for _ in range(n_samples):
        params = pnp.array(rng.uniform(0, 2 * np.pi, size=n_p), requires_grad=True)
        g = grad_fn(params)
        g0_samples.append(float(g[0]))

    return float(np.var(g0_samples))


def gradient_variance_all(ansatz_name, n_qubits, n_layers, noise_preset="noiseless",
                          n_samples=50, seed=0, cost="z0"):
    """
    Var_theta[dC/dtheta_k] averaged over ALL k (not just theta_0).
    cost = "z0"    : C = <Z_0>
    cost = "local" : C = (1/n) sum_i <Z_i>   (standard local cost)
    Returns dict with mean over parameters, min and max over parameters.
    """
    from pennylane import numpy as pnp

    rng = np.random.default_rng(seed)
    n_p = n_params(n_qubits, n_layers)
    if cost == "z0":
        obs = qml.PauliZ(0)
    elif cost == "local":
        obs = qml.Hamiltonian([1.0 / n_qubits] * n_qubits,
                              [qml.PauliZ(q) for q in range(n_qubits)])
    else:
        raise ValueError(cost)
    qnode = _make_expval_qnode(ansatz_name, n_qubits, n_layers, noise_preset, observable=obs)
    grad_fn = qml.grad(qnode, argnums=0)

    G = np.empty((n_samples, n_p))
    for s in range(n_samples):
        params = pnp.array(rng.uniform(0, 2 * np.pi, size=n_p), requires_grad=True)
        G[s] = np.array(grad_fn(params), dtype=float)
    v = G.var(axis=0)
    return {"mean": float(v.mean()), "min": float(v.min()), "max": float(v.max())}


# ---------------------------------------------------------------------------
# Metric 4: Function output variance
# ---------------------------------------------------------------------------
def function_output_variance(ansatz_name, n_qubits, n_layers, noise_preset="noiseless",
                             n_data=30, seed=0):
    rng = np.random.default_rng(seed)
    qnode = _make_data_qnode(ansatz_name, n_qubits, n_layers, noise_preset)
    n_p = n_params(n_qubits, n_layers)
    params = rng.uniform(0, 2 * np.pi, size=n_p)
    X = rng.uniform(0, np.pi, size=(n_data, n_qubits))
    outs = np.array([float(qnode(x, params)) for x in X])
    return float(np.var(outs))


# ---------------------------------------------------------------------------
# Metric 5: QFIM
# ---------------------------------------------------------------------------
def qfim_metrics(ansatz_name, n_qubits, n_layers, noise_preset="noiseless",
                 n_samples=15, n_data_for_eff_dim=1000, seed=0,
                 gamma=1.0, approx="block-diag"):
    """
    Returns dict with 'trace_F' (mean QFIM trace) and 'eff_dim' (Abbas et al. 2021).
    Uses block-diagonal Fubini-Study metric (factor of 4 for QFIM).
    """
    from pennylane import numpy as pnp

    rng = np.random.default_rng(seed)
    qnode = _make_expval_qnode(ansatz_name, n_qubits, n_layers, noise_preset)
    n_p = n_params(n_qubits, n_layers)

    traces = []
    fishers = []
    for _ in range(n_samples):
        params = pnp.array(rng.uniform(0, 2 * np.pi, size=n_p), requires_grad=True)
        try:
            F = qml.metric_tensor(qnode, approx=approx)(params)
            F = np.array(F)
            F_qfim = 4.0 * F
            traces.append(float(np.trace(F_qfim)))
            fishers.append(F_qfim)
        except Exception:
            continue

    if len(fishers) == 0:
        return {"trace_F": float("nan"), "eff_dim": float("nan")}

    trace_F = float(np.mean(traces))

    M = n_data_for_eff_dim
    kappa = gamma * M / (2 * np.pi * np.log(max(M, 2)))
    if kappa <= 1.0:
        return {"trace_F": trace_F, "eff_dim": float("nan")}

    sqrt_dets = []
    for F_qfim in fishers:
        norm = n_p / max(np.trace(F_qfim), 1e-12)
        F_hat = norm * F_qfim
        F_hat_sym = (F_hat + F_hat.T) / 2
        eigvals = np.linalg.eigvalsh(F_hat_sym)
        eigvals = np.clip(eigvals, 0.0, None)
        log_det = float(np.sum(np.log1p(kappa * eigvals)))
        sqrt_dets.append(0.5 * log_det)

    sd = np.array(sqrt_dets)
    smax = sd.max()
    log_mean_sqrt_det = smax + np.log(np.mean(np.exp(sd - smax)))
    eff_dim = float(2.0 * log_mean_sqrt_det / np.log(kappa))
    eff_dim = max(0.0, eff_dim)
    return {"trace_F": trace_F, "eff_dim": eff_dim}


In [ ]:
%%writefile analysis.py
"""
analysis.py
===========
Statistical analysis utilities:
    - bootstrap_ci      : bootstrap confidence intervals for any statistic.
    - fit_power_law     : log-linear fit y ~ x^alpha with bootstrap CI on alpha.
    - fit_exp_decay     : log-linear fit y ~ exp(-beta * x) with bootstrap CI on beta.
    - aggregate_runs    : repeated metric evaluation with mean, std, 95% CI.
"""

import numpy as np
from scipy.stats import linregress


def bootstrap_ci(values, statistic=np.mean, n_boot=1000, ci=95, seed=0):
    """Return (point_estimate, lo, hi) at the requested CI level."""
    rng = np.random.default_rng(seed)
    values = np.asarray(values, dtype=float)
    if values.size == 0:
        return float("nan"), float("nan"), float("nan")
    point = float(statistic(values))
    n = len(values)
    boots = np.empty(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, n, size=n)
        boots[b] = statistic(values[idx])
    lo = float(np.percentile(boots, (100 - ci) / 2))
    hi = float(np.percentile(boots, 100 - (100 - ci) / 2))
    return point, lo, hi


def fit_power_law(x, y, n_boot=1000, seed=0):
    """
    Fit y = c * x^alpha by linear regression on log-log data.
    Returns dict with alpha (slope), intercept (log c), R^2, alpha_lo, alpha_hi (95% CI).
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    mask = (x > 0) & (y > 0) & np.isfinite(x) & np.isfinite(y)
    x, y = x[mask], y[mask]
    if len(x) < 3:
        return {"alpha": float("nan"), "intercept": float("nan"),
                "r2": float("nan"), "alpha_lo": float("nan"), "alpha_hi": float("nan")}

    logx, logy = np.log(x), np.log(y)
    res = linregress(logx, logy)
    alpha = float(res.slope)
    intercept = float(res.intercept)
    r2 = float(res.rvalue ** 2)

    rng = np.random.default_rng(seed)
    n = len(x)
    boot_alphas = np.empty(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, n, size=n)
        if len(np.unique(idx)) < 2:
            boot_alphas[b] = alpha
            continue
        r = linregress(logx[idx], logy[idx])
        boot_alphas[b] = r.slope
    alpha_lo = float(np.percentile(boot_alphas, 2.5))
    alpha_hi = float(np.percentile(boot_alphas, 97.5))

    return {"alpha": alpha, "intercept": intercept, "r2": r2,
            "alpha_lo": alpha_lo, "alpha_hi": alpha_hi}


def fit_exp_decay(x, y, n_boot=1000, seed=0):
    """
    Fit y = c * exp(-beta * x) by linear regression on log y vs x.
    Returns dict with beta, intercept, R^2, beta_lo, beta_hi.
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = (y > 0) & np.isfinite(x) & np.isfinite(y)
    x, y = x[mask], y[mask]
    if len(x) < 3:
        return {"beta": float("nan"), "intercept": float("nan"),
                "r2": float("nan"), "beta_lo": float("nan"), "beta_hi": float("nan")}

    logy = np.log(y)
    res = linregress(x, logy)
    beta = float(-res.slope)
    intercept = float(res.intercept)
    r2 = float(res.rvalue ** 2)

    rng = np.random.default_rng(seed)
    n = len(x)
    boot_betas = np.empty(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, n, size=n)
        if len(np.unique(idx)) < 2:
            boot_betas[b] = beta
            continue
        r = linregress(x[idx], logy[idx])
        boot_betas[b] = -r.slope
    beta_lo = float(np.percentile(boot_betas, 2.5))
    beta_hi = float(np.percentile(boot_betas, 97.5))

    return {"beta": beta, "intercept": intercept, "r2": r2,
            "beta_lo": beta_lo, "beta_hi": beta_hi}


def aggregate_runs(metric_fn, n_runs=30, base_seed=0, **metric_kwargs):
    """
    Run metric_fn n_runs times with different seeds, return mean, std, 95% CI.
    metric_fn must accept a `seed` keyword.
    """
    vals = []
    for r in range(n_runs):
        v = metric_fn(seed=base_seed + r, **metric_kwargs)
        if isinstance(v, dict):
            vals.append(v)
        else:
            vals.append(float(v))

    if isinstance(vals[0], dict):
        keys = vals[0].keys()
        out = {}
        for k in keys:
            arr = np.array([v[k] for v in vals], dtype=float)
            arr = arr[np.isfinite(arr)]
            if arr.size == 0:
                out[k] = {"mean": float("nan"), "std": float("nan"),
                          "lo": float("nan"), "hi": float("nan")}
                continue
            mean, lo, hi = bootstrap_ci(arr, statistic=np.mean, n_boot=1000, seed=base_seed)
            out[k] = {"mean": mean, "std": float(np.std(arr, ddof=1)) if arr.size > 1 else 0.0,
                      "lo": lo, "hi": hi}
        return out

    arr = np.array(vals, dtype=float)
    arr = arr[np.isfinite(arr)]
    mean, lo, hi = bootstrap_ci(arr, statistic=np.mean, n_boot=1000, seed=base_seed)
    return {"mean": mean,
            "std": float(np.std(arr, ddof=1)) if arr.size > 1 else 0.0,
            "lo": lo, "hi": hi, "n": int(arr.size)}


In [ ]:
%%writefile run_fixed_rerun.py
"""
run_fixed_rerun.py
==================
Rerun after the Nov 2026 audit fixes:
  * noise applied after EVERY layer (ansatz layer_hook)
  * noisy and noiseless cells use the SAME seeds and the SAME n_runs (paired)
  * gradient variance averaged over ALL parameters, for two costs:
        z0    : C = <Z_0>
        local : C = (1/n) sum_i <Z_i>
    (the old theta_0 / Z_0 diagnostic is kept as grad_var_theta0 for continuity)
  * expressivity: Haar KL against the REAL Haar ensemble + frame-potential ratio
  * kernel rank with n_data large enough not to cap the rank (default 64)
  * heavy_hex_true: real IBM Falcon-27 coupling-map subgraphs

Groups
------
  noisy     : n = 2..6, all noise presets incl. noiseless (paired), gradients only
  noiseless : n = 2..N_MAX, gradients + expressivity + kernel rank

Usage
-----
  python run_fixed_rerun.py --group noisy --out outputs_fixed
  python run_fixed_rerun.py --group noiseless --nmax 12 --out outputs_fixed
  python run_fixed_rerun.py --group noisy --quick          # small preview run
Re-running resumes: finished cells are skipped.
"""

import argparse
import multiprocessing as mp
import os
import time
import warnings

# One BLAS thread per worker process; otherwise 4 workers x N threads oversubscribe the CPUs.
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_v, "1")

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

TOPOS = ["linear", "ring", "brick", "all_to_all", "star", "heavy_hex", "heavy_hex_true"]
NOISES = ["noiseless", "depolarizing_realistic", "depolarizing_strong",
          "amplitude_realistic", "amplitude_strong"]


def grad_stats(ansatz, n, L, noise, n_samples, seed):
    """One run: per-parameter gradient variances for z0 and local costs,
    computed from a single Jacobian of all <Z_i>."""
    import pennylane as qml
    from pennylane import numpy as pnp
    from ansatze import ANSATZE
    from noise import make_device, is_noiseless, apply_layer_noise

    dev = make_device(n, noise)
    hook = None if is_noiseless(noise) else (lambda: apply_layer_noise(noise, n))

    @qml.qnode(dev, interface="autograd")
    def circ(p):
        ANSATZE[ansatz](p, n, L, layer_hook=hook)
        return [qml.expval(qml.PauliZ(q)) for q in range(n)]

    def vec(p):
        return pnp.stack(circ(p))

    jac = qml.jacobian(vec)
    rng = np.random.default_rng(seed)
    n_p = n * L
    J = np.empty((n_samples, n, n_p))
    for s in range(n_samples):
        p = pnp.array(rng.uniform(0, 2 * np.pi, n_p), requires_grad=True)
        J[s] = np.array(jac(p), dtype=float).reshape(n, n_p)
    g_z0 = J[:, 0, :]
    g_loc = J.mean(axis=1)
    v_z0, v_loc = g_z0.var(axis=0), g_loc.var(axis=0)
    return {
        "grad_var_theta0": float(v_z0[0]),
        "grad_var_z0_all": float(v_z0.mean()),
        "grad_var_local_all": float(v_loc.mean()),
        "grad_var_local_min": float(v_loc.min()),
    }


def run_cell(task):
    import sys
    sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
    warnings.filterwarnings("ignore")
    from analysis import bootstrap_ci
    from metrics import frame_potential_ratio, haar_kl_divergence, effective_kernel_rank

    group, ansatz, noise, n, L, n_runs, n_samples, n_data = task
    per_run = {}
    t0 = time.time()
    for r in range(n_runs):
        out = grad_stats(ansatz, n, L, noise, n_samples, seed=r)
        if group == "noiseless":
            out["frame_potential_ratio"] = frame_potential_ratio(ansatz, n, L, noise, n_samples=60, seed=r)
            out["haar_kl_real"] = haar_kl_divergence(ansatz, n, L, noise, n_samples=60, seed=r, reference="real")
            out["kernel_rank"] = effective_kernel_rank(ansatz, n, L, noise, n_data=n_data, seed=r)
        for k, v in out.items():
            per_run.setdefault(k, []).append(v)
    rows = []
    for metric, vals in per_run.items():
        arr = np.array(vals, dtype=float)
        mean, lo, hi = bootstrap_ci(arr, n_boot=1000, seed=0)
        rows.append({"ansatz": ansatz, "noise": noise, "n_qubits": n, "n_layers": L,
                     "n_params": n * L, "metric": metric, "mean": mean,
                     "std": float(arr.std(ddof=1)) if arr.size > 1 else 0.0,
                     "lo": lo, "hi": hi, "n_runs": n_runs, "n_samples": n_samples,
                     "seeds": f"0-{n_runs - 1}", "n_data": n_data if group == "noiseless" else "",
                     "seconds": round(time.time() - t0, 1)})
    return rows


def build_tasks(args):
    tasks = []
    if args.group == "noisy":
        ns = range(2, (min(5, args.nmax_noisy) if args.quick else args.nmax_noisy) + 1)
        Ls = [1, 4, 8] if args.quick else range(1, 9)
        for a in TOPOS:
            for noise in NOISES:
                for n in ns:
                    for L in Ls:
                        tasks.append(("noisy", a, noise, n, L, args.runs, args.samples, 0))
    else:
        ns = range(2, args.nmax + 1)
        Ls = [1, 4, 8] if args.quick else range(1, 9)
        for a in TOPOS:
            for n in ns:
                for L in Ls:
                    tasks.append(("noiseless", a, "noiseless", n, L, args.runs, args.samples, args.ndata))
    return tasks


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--group", choices=["noisy", "noiseless"], required=True)
    ap.add_argument("--out", default="outputs_fixed")
    ap.add_argument("--runs", type=int, default=20)
    ap.add_argument("--samples", type=int, default=50)
    ap.add_argument("--ndata", type=int, default=64)
    ap.add_argument("--nmax", type=int, default=10)
    ap.add_argument("--quick", action="store_true", help="small preview grid")
    ap.add_argument("--procs", type=int, default=max(1, mp.cpu_count() - 1))
    ap.add_argument("--max-hours", type=float, default=0,
                    help="stop cleanly after this many hours (0 = no limit); rerun to resume")
    ap.add_argument("--nmax-noisy", type=int, default=6)
    args = ap.parse_args()
    if args.quick and args.runs == 20:
        args.runs, args.samples = 5, 30

    os.makedirs(args.out, exist_ok=True)
    tag = f"{args.group}{'_quick' if args.quick else ''}"
    cells = os.path.join(args.out, f"cells_{tag}.csv")
    done = set()
    if os.path.exists(cells):
        d = pd.read_csv(cells)
        done = set(zip(d.ansatz, d.noise, d.n_qubits, d.n_layers))
    tasks = [t for t in build_tasks(args) if (t[1], t[2], t[3], t[4]) not in done]
    tasks.sort(key=lambda t: t[3] * t[4] * (3 if t[2] != "noiseless" else 1))
    print(f"{len(tasks)} cells to run ({len(done)} already done), {args.procs} processes", flush=True)

    t0 = time.time()
    stopped = False
    pool = mp.Pool(args.procs)
    try:
        for i, rows in enumerate(pool.imap_unordered(run_cell, tasks, chunksize=1), 1):
            pd.DataFrame(rows).to_csv(cells, mode="a", header=not os.path.exists(cells), index=False)
            el = time.time() - t0
            if i % 10 == 0 or i == len(tasks):
                print(f"[{i}/{len(tasks)}] {el/60:.1f} min elapsed, eta {el/i*(len(tasks)-i)/60:.1f} min", flush=True)
            if args.max_hours and el > args.max_hours * 3600:
                stopped = True
                print(f"Time budget of {args.max_hours} h reached after {i} cells; "
                      f"saved results are kept. Run again to continue.", flush=True)
                break
    finally:
        if stopped:
            pool.terminate()
        else:
            pool.close()
        pool.join()
    print(("stopped early: " if stopped else "done: ") + cells, flush=True)


if __name__ == "__main__":
    mp.freeze_support()
    main()


In [ ]:
%%writefile test_fixes.py
"""
test_fixes.py
=============
Checks for the Nov 2026 fixes. Run:  python -m pytest -q test_fixes.py
(or just: python test_fixes.py)
"""
import numpy as np
import pennylane as qml

from ansatze import ANSATZE, ANSATZE_WITH_DATA, TOPOLOGIES, FALCON27_EDGES, _falcon_subgraph
from metrics import (_make_expval_qnode, _make_state_qnode, _haar_fidelities_real,
                     haar_frame_potential, frame_potential_ratio, gradient_variance_all,
                     effective_kernel_rank, haar_kl_divergence)
from noise import apply_layer_noise


def _ops(fn, *args, **kw):
    return qml.tape.make_qscript(fn)(*args, **kw).operations


def test_noise_applied_every_layer():
    n, L = 3, 4
    hook = lambda: apply_layer_noise("depolarizing_strong", n)
    for name in TOPOLOGIES:
        ops = _ops(ANSATZE[name], np.zeros(n * L), n, L, layer_hook=hook)
        k = sum(isinstance(o, qml.DepolarizingChannel) for o in ops)
        assert k == n * L, (name, k)


def test_noisy_qnode_differs_from_end_only_noise():
    """Per-layer noise must damp <Z0> more than a single channel at the end."""
    n, L = 2, 4
    rng = np.random.default_rng(1)
    p = rng.uniform(0, 2 * np.pi, n * L)
    per_layer = float(_make_expval_qnode("linear", n, L, "depolarizing_strong")(p))
    dev = qml.device("default.mixed", wires=n)

    @qml.qnode(dev)
    def end_only(params):
        ANSATZE["linear"](params, n, L)
        apply_layer_noise("depolarizing_strong", n)
        return qml.expval(qml.PauliZ(0))
    assert abs(per_layer - float(end_only(p))) > 1e-6


def test_noiseless_equals_old_circuit():
    """With no hook the circuits are unchanged from the original code."""
    n, L = 3, 2
    p = np.linspace(0.1, 1.2, n * L)
    dev = qml.device("default.qubit", wires=n)

    @qml.qnode(dev)
    def old_linear(params):
        idx = 0
        for _ in range(L):
            for q in range(n):
                qml.RY(params[idx], wires=q); idx += 1
            for q in range(n - 1):
                qml.CNOT(wires=[q, q + 1])
        return qml.expval(qml.PauliZ(0))
    assert np.isclose(float(_make_expval_qnode("linear", n, L, "noiseless")(p)), float(old_linear(p)))


def test_states_are_real():
    """RY + CNOT/CZ circuits give real amplitudes, so the real-Haar reference is right."""
    n, L = 4, 3
    rng = np.random.default_rng(2)
    for name in TOPOLOGIES:
        qnode, mode = _make_state_qnode(name, n, L, "noiseless")
        st = np.array(qnode(rng.uniform(0, 2 * np.pi, n * L)))
        assert np.max(np.abs(st.imag)) < 1e-10, name


def test_real_haar_sampler():
    rng = np.random.default_rng(3)
    for n in (2, 4, 6):
        d = 2 ** n
        f = _haar_fidelities_real(n, 200000, rng)
        assert abs(f.mean() * d - 1) < 0.03
        assert abs(np.mean(f ** 2) / haar_frame_potential(n) - 1) < 0.05


def test_falcon_subgraph():
    assert len(FALCON27_EDGES) == 28
    nodes = {v for e in FALCON27_EDGES for v in e}
    assert nodes == set(range(27))
    deg = {}
    for a, b in FALCON27_EDGES:
        deg[a] = deg.get(a, 0) + 1; deg[b] = deg.get(b, 0) + 1
    assert max(deg.values()) <= 3
    for n in range(2, 11):
        E = _falcon_subgraph(n)
        adj = {i: set() for i in range(n)}
        for a, b in E:
            adj[a].add(b); adj[b].add(a)
        seen, stack = {0}, [0]
        while stack:
            v = stack.pop()
            for w in adj[v] - seen:
                seen.add(w); stack.append(w)
        assert len(seen) == n, n
        assert max(len(s) for s in adj.values()) <= 3


def test_new_metrics_run():
    for name in ("linear", "star", "heavy_hex_true"):
        r = gradient_variance_all(name, 3, 2, "noiseless", n_samples=5, cost="local")
        assert r["min"] <= r["mean"] <= r["max"]
        assert frame_potential_ratio(name, 3, 2, n_samples=10) > 0
        assert effective_kernel_rank(name, 3, 2, n_data=8) > 0
        assert haar_kl_divergence(name, 3, 2, n_samples=10) >= 0
    r = gradient_variance_all("ring", 2, 2, "amplitude_strong", n_samples=3, cost="z0")
    assert np.isfinite(r["mean"])


if __name__ == "__main__":
    import sys, inspect
    fails = 0
    for name, fn in list(globals().items()):
        if name.startswith("test_") and inspect.isfunction(fn):
            try:
                fn(); print("PASS", name)
            except Exception as e:
                fails += 1; print("FAIL", name, repr(e))
    sys.exit(1 if fails else 0)


## Tests (must all PASS)

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, 'test_fixes.py'], capture_output=True, text=True)
print(r.stdout[-3000:], r.stderr[-3000:])
assert r.returncode == 0, 'Tests failed: stop and send this output back.'

## Resume: copy finished cells from a previous version (if added as input)

In [ ]:
import glob, os, shutil
import pandas as pd
os.makedirs('outputs_fixed', exist_ok=True)
for src in glob.glob('/kaggle/input/**/outputs_fixed/cells_*.csv', recursive=True):
    dst = os.path.join('outputs_fixed', os.path.basename(src))
    old = pd.read_csv(src)
    if os.path.exists(dst):
        old = pd.concat([pd.read_csv(dst), old])
    old.drop_duplicates(['ansatz', 'noise', 'n_qubits', 'n_layers', 'metric']).to_csv(dst, index=False)
    print('resumed from', src)
print(sorted(os.listdir('outputs_fixed')))

## Run the stage

In [ ]:
if SMOKE:
    extra = ['--quick', '--runs', '2', '--samples', '5', '--nmax', '4', '--nmax-noisy', '3']
else:
    extra = ['--nmax', '12'] if STAGE == 'noiseless' else []
cmd = [sys.executable, 'run_fixed_rerun.py', '--group', STAGE, '--out', 'outputs_fixed',
       '--procs', str(os.cpu_count()), '--max-hours', str(MAX_HOURS)] + extra
print(' '.join(cmd))
p = subprocess.run(cmd, text=True)
assert p.returncode == 0, 'Runner failed: send the log above back.'

## Summary and package

In [ ]:
for f in sorted(glob.glob('outputs_fixed/cells_*.csv')):
    d = pd.read_csv(f)
    n = len(d.drop_duplicates(['ansatz', 'noise', 'n_qubits', 'n_layers']))
    print(f'{f}: {n} cells, {len(d)} rows, metrics: {sorted(d.metric.unique())}')
shutil.make_archive('outputs_fixed', 'zip', '.', 'outputs_fixed')
print('wrote outputs_fixed.zip', os.path.getsize('outputs_fixed.zip'), 'bytes')